In [0]:
%sql
CREATE OR REPLACE TABLE `medallion_architecture`.`gold`.`gold_facility_location` (
  -- Identity
  `location_id` STRING COMMENT 'Stable internal location ID (using unique_id as 1\:1 mapping)',
  `facility_id` STRING COMMENT 'Foreign key to gold_facility.facility_id',
  
  -- Address Components
  `address_full` STRING COMMENT 'Best full address string',
  `address_line_1` STRING COMMENT 'Street or building-level address',
  `address_line_2` STRING COMMENT 'Additional address details',
  `address_line_3` STRING COMMENT 'Additional address details',
  `city` STRING COMMENT 'City or locality',
  `district` STRING COMMENT 'District, county, or equivalent',
  `state_region` STRING COMMENT 'State, province, or region',
  `postal_code` STRING COMMENT 'Postal or ZIP code',
  `country` STRING COMMENT 'Country',
  
  -- Coordinates
  `latitude` DOUBLE COMMENT 'Best latitude',
  `longitude` DOUBLE COMMENT 'Best longitude',
  `coordinates_json` STRING COMMENT 'GeoJSON coordinates string',
  
  -- Quality & Metadata
  `geo_confidence_score` DOUBLE COMMENT 'Confidence in coordinates and address match (0-1)',
  `is_primary_location` BOOLEAN COMMENT 'Whether this is the primary facility location',
  `location_quality_flags` ARRAY<STRING> COMMENT 'Missing coordinates, conflicting coordinates, partial address, etc.',
  `last_verified_at` TIMESTAMP COMMENT 'Most recent verification date for this location',
  
  -- Audit
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  `updated_at` TIMESTAMP COMMENT 'Record last update timestamp'
)
COMMENT 'Gold layer location table for facility addresses and coordinates. One row per facility location.';

In [0]:
%sql
INSERT INTO `medallion_architecture`.`gold`.`gold_facility_location`
SELECT
  -- Identity: Use unique_id as location_id (1\:1 mapping with facility)
  `unique_id` AS `location_id`,
  `unique_id` AS `facility_id`,
  
  -- Address Full: Concatenate all address components
  TRIM(
    CONCAT_WS(', ',
      NULLIF(TRIM(`address_line1`), ''),
      NULLIF(TRIM(`address_line2`), ''),
      NULLIF(TRIM(`address_line3`), ''),
      NULLIF(TRIM(`address_city`), ''),
      NULLIF(TRIM(`address_state_or_region`), ''),
      NULLIF(TRIM(`address_zip_or_postcode`), ''),
      CASE 
        WHEN `address_country` = 'India' THEN 'India'
        WHEN `address_country` IS NOT NULL 
          AND `address_country` NOT LIKE '%coordinates%'
          AND `address_country` NOT LIKE '%type%'
          AND NOT `address_country` RLIKE '^[0-9.]+$'
          AND `address_country` != 'kie' THEN TRIM(`address_country`)
        ELSE NULL
      END
    )
  ) AS `address_full`,
  
  -- Address Components
  TRIM(`address_line1`) AS `address_line_1`,
  TRIM(`address_line2`) AS `address_line_2`,
  TRIM(`address_line3`) AS `address_line_3`,
  TRIM(`address_city`) AS `city`,
  
  -- District: Not available in source, set to NULL
  CAST(NULL AS STRING) AS `district`,
  
  -- State/Region: Clean up garbage values
  CASE 
    WHEN `address_state_or_region` IS NOT NULL
      AND `address_state_or_region` NOT LIKE '%coordinates%'
      AND `address_state_or_region` NOT LIKE '%type%'
      AND NOT `address_state_or_region` RLIKE '^[0-9.]+$'
    THEN TRIM(`address_state_or_region`)
    ELSE NULL
  END AS `state_region`,
  
  TRIM(`address_zip_or_postcode`) AS `postal_code`,
  
  -- Country: Clean up garbage values
  CASE 
    WHEN `address_country` = 'India' THEN 'India'
    WHEN `address_country` IS NOT NULL 
      AND `address_country` NOT LIKE '%coordinates%'
      AND `address_country` NOT LIKE '%type%'
      AND NOT `address_country` RLIKE '^[0-9.]+$'
      AND `address_country` != 'kie'
    THEN TRIM(`address_country`)
    ELSE NULL
  END AS `country`,
  
  -- Coordinates
  `latitude`,
  `longitude`,
  `coordinates` AS `coordinates_json`,
  
  -- Geo Confidence Score: Calculate based on data completeness
  (
    CASE WHEN `latitude` IS NOT NULL AND `longitude` IS NOT NULL THEN 0.4 ELSE 0.0 END +
    CASE WHEN `address_city` IS NOT NULL THEN 0.2 ELSE 0.0 END +
    CASE WHEN `address_state_or_region` IS NOT NULL 
      AND `address_state_or_region` NOT LIKE '%coordinates%' THEN 0.2 ELSE 0.0 END +
    CASE WHEN `address_zip_or_postcode` IS NOT NULL THEN 0.1 ELSE 0.0 END +
    CASE WHEN `address_line1` IS NOT NULL THEN 0.1 ELSE 0.0 END
  ) AS `geo_confidence_score`,
  
  -- Primary Location: All locations are primary (1\:1 mapping)
  TRUE AS `is_primary_location`,
  
  -- Location Quality Flags: Identify issues
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `latitude` IS NULL OR `longitude` IS NULL THEN 'missing_coordinates' ELSE NULL END,
    CASE WHEN `address_city` IS NULL THEN 'missing_city' ELSE NULL END,
    CASE WHEN `address_state_or_region` IS NULL 
      OR `address_state_or_region` LIKE '%coordinates%' THEN 'missing_state' ELSE NULL END,
    CASE WHEN `address_zip_or_postcode` IS NULL THEN 'missing_postal_code' ELSE NULL END,
    CASE WHEN `address_line1` IS NULL THEN 'missing_street_address' ELSE NULL END,
    CASE WHEN `address_country` IS NULL 
      OR `address_country` LIKE '%coordinates%'
      OR `address_country` RLIKE '^[0-9.]+$' THEN 'invalid_country' ELSE NULL END,
    CASE WHEN `coordinates` IS NULL THEN 'missing_geojson' ELSE NULL END
  )) AS `location_quality_flags`,
  
  -- Last Verified: Use most recent date available
  COALESCE(
    `most_recent_social_media_post_date`,
    `recency_of_page_update`,
    `silver_ingested_at`
  ) AS `last_verified_at`,
  
  -- Audit timestamps
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`

FROM `medallion_architecture`.`silver`.`facilities`;

num_affected_rows,num_inserted_rows
10077,10077


In [0]:
%sql

SELECT
  'location_id' as column_name,
  COUNT(*) as total_records,
  COUNT(`location_id`) as non_null_count,
  COUNT(*) - COUNT(`location_id`) as null_count,
  ROUND(COUNT(`location_id`) / COUNT(*), 4) as completeness_pct,
  ROUND((COUNT(*) - COUNT(`location_id`)) / COUNT(*), 4) as missing_pct
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'facility_id',
  COUNT(*),
  COUNT(`facility_id`),
  COUNT(*) - COUNT(`facility_id`),
  ROUND(COUNT(`facility_id`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`facility_id`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'address_full',
  COUNT(*),
  COUNT(`address_full`),
  COUNT(*) - COUNT(`address_full`),
  ROUND(COUNT(`address_full`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`address_full`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'address_line_1',
  COUNT(*),
  COUNT(`address_line_1`),
  COUNT(*) - COUNT(`address_line_1`),
  ROUND(COUNT(`address_line_1`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`address_line_1`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'address_line_2',
  COUNT(*),
  COUNT(`address_line_2`),
  COUNT(*) - COUNT(`address_line_2`),
  ROUND(COUNT(`address_line_2`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`address_line_2`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'address_line_3',
  COUNT(*),
  COUNT(`address_line_3`),
  COUNT(*) - COUNT(`address_line_3`),
  ROUND(COUNT(`address_line_3`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`address_line_3`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'city',
  COUNT(*),
  COUNT(`city`),
  COUNT(*) - COUNT(`city`),
  ROUND(COUNT(`city`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`city`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'district',
  COUNT(*),
  COUNT(`district`),
  COUNT(*) - COUNT(`district`),
  ROUND(COUNT(`district`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`district`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'state_region',
  COUNT(*),
  COUNT(`state_region`),
  COUNT(*) - COUNT(`state_region`),
  ROUND(COUNT(`state_region`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`state_region`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'postal_code',
  COUNT(*),
  COUNT(`postal_code`),
  COUNT(*) - COUNT(`postal_code`),
  ROUND(COUNT(`postal_code`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`postal_code`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'country',
  COUNT(*),
  COUNT(`country`),
  COUNT(*) - COUNT(`country`),
  ROUND(COUNT(`country`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`country`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'latitude',
  COUNT(*),
  COUNT(`latitude`),
  COUNT(*) - COUNT(`latitude`),
  ROUND(COUNT(`latitude`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`latitude`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'longitude',
  COUNT(*),
  COUNT(`longitude`),
  COUNT(*) - COUNT(`longitude`),
  ROUND(COUNT(`longitude`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`longitude`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'coordinates_json',
  COUNT(*),
  COUNT(`coordinates_json`),
  COUNT(*) - COUNT(`coordinates_json`),
  ROUND(COUNT(`coordinates_json`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`coordinates_json`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'geo_confidence_score',
  COUNT(*),
  COUNT(`geo_confidence_score`),
  COUNT(*) - COUNT(`geo_confidence_score`),
  ROUND(COUNT(`geo_confidence_score`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`geo_confidence_score`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'is_primary_location',
  COUNT(*),
  COUNT(`is_primary_location`),
  COUNT(*) - COUNT(`is_primary_location`),
  ROUND(COUNT(`is_primary_location`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`is_primary_location`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'location_quality_flags',
  COUNT(*),
  COUNT(`location_quality_flags`),
  COUNT(*) - COUNT(`location_quality_flags`),
  ROUND(COUNT(`location_quality_flags`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`location_quality_flags`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
UNION ALL
SELECT
  'last_verified_at',
  COUNT(*),
  COUNT(`last_verified_at`),
  COUNT(*) - COUNT(`last_verified_at`),
  ROUND(COUNT(`last_verified_at`) / COUNT(*), 4),
  ROUND((COUNT(*) - COUNT(`last_verified_at`)) / COUNT(*), 4)
FROM
  `medallion_architecture`.`gold`.`gold_facility_location`
ORDER BY
  missing_pct DESC

column_name,total_records,non_null_count,null_count,completeness_pct,missing_pct
district,10077,0,10077,0.0,1.0
address_line_3,10077,7031,3046,0.6977,0.3023
address_line_2,10077,9792,285,0.9717,0.0283
latitude,10077,9959,118,0.9883,0.0117
longitude,10077,9959,118,0.9883,0.0117
coordinates_json,10077,9960,117,0.9884,0.0116
country,10077,9991,86,0.9915,0.0085
state_region,10077,9997,80,0.9921,0.0079
address_line_1,10077,10008,69,0.9932,0.0068
postal_code,10077,10011,66,0.9935,0.0065
